In [1]:
import pandas as pd

In [2]:
# 1. Load your filtered fMRI CSV and the re-exported MRI CSV (with Image ID & Acq Date)
fmri_df = pd.read_csv("../../results/dataset_creation/only_bold_fmri.csv")
smri_df = pd.read_csv("../../data/smri-adni-search-result.csv")
qc_df = pd.read_csv("../../data/MRIQC_19Oct2026.csv", low_memory=False)

In [3]:
# Create integer LONI Image IDs for exact joining
fmri_df["raw_id"] = (
    fmri_df["Image Data ID"].astype(str).str.replace("I", "").astype(int)
)
smri_df["raw_id"] = smri_df["Image ID"].astype(int)
qc_df["LONIImage"] = qc_df["LONIImage"].astype(int)

In [4]:
# 2. Enrich fMRI with MRIQC header metadata
qc_cols = [
    "LONIImage",
    "VISCODE2",
    "StudyDate",
    "SeriesTime",
    "SeriesNumber",
    "SeriesType",
    "MRIProtocolPhase",
    "ScannerManufacturer",
    "ScannerModel",
    "MagneticFieldStrength",
    "Acceleration",
    "NumberVolumes",
    "SlicesPerVolume",
    "SliceThickness",
    "LONIStudy",
]
fmri_qc = fmri_df.merge(
    qc_df[qc_cols], left_on="raw_id", right_on="LONIImage", how="left"
)

In [5]:
fmri_qc

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,SeriesType,MRIProtocolPhase,ScannerManufacturer,ScannerModel,MagneticFieldStrength,Acceleration,NumberVolumes,SlicesPerVolume,SliceThickness,LONIStudy
0,I238623,002_S_0295,CN,M,90,v06,fMRI,Resting State fMRI,6/02/2011,238623,...,BOLD,ADNIGO/ADNI2,Philips,Intera,3.0,Unaccelerated,140.0,48.0,3.31,34310.0
1,I303069,002_S_0295,CN,M,91,v11,fMRI,Resting State fMRI,5/10/2012,303069,...,BOLD,ADNIGO/ADNI2,Philips,Intera,3.0,Unaccelerated,140.0,48.0,3.31,45566.0
2,I1221056,002_S_0413,CN,F,90,y2,fMRI,Axial MB rsfMRI (Eyes Open),8/27/2019,1221056,...,BOLD,ADNI3,Siemens,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,142330.0
3,I863058,002_S_0413,CN,F,87,init,fMRI,Axial rsfMRI (Eyes Open),6/21/2017,863058,...,BOLD,ADNI3,Siemens,Prisma_fit,3.0,Accelerated,197.0,48.0,3.40,112339.0
4,I240811,002_S_0413,CN,F,82,v06,fMRI,Resting State fMRI,6/16/2011,240811,...,BOLD,ADNIGO/ADNI2,Philips,Intera,3.0,Unaccelerated,140.0,48.0,3.31,34751.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3360,I1553010,941_S_7051,CN,M,64,sc,fMRI,Axial MB rsfMRI (Eyes Open),3/07/2022,1553010,...,BOLD,ADNI3,Siemens,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,193272.0
3361,I1588336,941_S_7074,CN,M,71,sc,fMRI,Axial MB rsfMRI (Eyes Open),5/03/2022,1588336,...,BOLD,ADNI3,Siemens,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,211723.0
3362,I1600185,941_S_7085,MCI,F,70,sc,fMRI,Axial MB rsfMRI (Eyes Open),7/01/2022,1600185,...,BOLD,ADNI3,Siemens,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,213547.0
3363,I1591326,941_S_7087,CN,M,67,sc,fMRI,Axial MB rsfMRI (Eyes Open),6/15/2022,1591326,...,BOLD,ADNI3,Siemens,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,212360.0


In [6]:
# 3. Filter out non-BOLD (MRS) and corrupted/aborted fMRI runs (< 90 timepoints)
#    (Preserve the 4 ungrouped 2D-slice scans where NumberVolumes == 1 but SlicesPerVolume >= 4000,
#     and preserve the 4 rows where MRIQC metadata is NaN so you can check them locally)
is_valid_type = fmri_qc["SeriesType"].isin(["BOLD"]) | fmri_qc["SeriesType"].isna()
has_enough_vols = (
    (fmri_qc["NumberVolumes"] >= 90)
    | ((fmri_qc["NumberVolumes"] == 1) & (fmri_qc["SlicesPerVolume"] >= 4000))
    | fmri_qc["NumberVolumes"].isna()
)

clean_fmri = fmri_qc[is_valid_type & has_enough_vols].copy()
print(f"fMRI rows before QC filtering: {len(fmri_qc)}")
print(
    f"fMRI rows after dropping MRS & aborted/0-vol runs: {len(clean_fmri)} (dropped {len(fmri_qc) - len(clean_fmri)} bad runs)"
)

fMRI rows before QC filtering: 3365
fMRI rows after dropping MRS & aborted/0-vol runs: 3309 (dropped 56 bad runs)


In [7]:
qc_df[qc_df["image_id"] == 1011355]

,image_id,ParticipantID,VISCODE2,StudyDate,SeriesTime,SeriesNumber,SeriesType,MRIProtocolPhase,SeriesDescription,ScannerManufacturer,...,NumberVolumes,SlicesPerVolume,SliceThickness,ReceiveCoilName,StudyInstanceUID,SeriesInstanceUID,LONIStudy,LONISeries,LONIImage,update_stamp
55875,1011355,019_S_4293,m78,2018-06-15,12:53:40,901,BOLD,ADNI3,Axial fcMRI (EYES OPEN),Philips,...,1,1,3.39,SENSE-Head-8,2.16.124.113543.6006.99.08378446856015111223,2.16.124.113543.6006.99.4288473413097688460,123923,696764,1011355,2026-03-23 12:51:16


In [8]:
fmri_qc[fmri_qc["SeriesType"].isna()]

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,SeriesType,MRIProtocolPhase,ScannerManufacturer,ScannerModel,MagneticFieldStrength,Acceleration,NumberVolumes,SlicesPerVolume,SliceThickness,LONIStudy
288,I249406,006_S_4150,CN,M,74,v02,fMRI,Resting State fMRI,8/08/2011,249406,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
427,I1248413,006_S_6277,CN,F,72,y1,fMRI,Axial fcMRI (EYES OPEN),4/25/2019,1248413,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
920,I1011356,019_S_4293,LMCI,M,76,y1,fMRI,Axial fcMRI (EYES OPEN),6/15/2018,1011356,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1363,I424806,031_S_4005,EMCI,M,74,v31,fMRI,Extended Resting State fMRI,5/13/2014,424806,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# 4. Deduplicate same-day fMRI runs per (Subject, Acq Date):
#    Prefer full volume count -> latest SeriesTime (replacement run) -> highest SeriesNumber
clean_fmri["Acq Date_dt"] = pd.to_datetime(clean_fmri["Acq Date"])
clean_fmri["vol_sort"] = clean_fmri["NumberVolumes"].fillna(140.0)
clean_fmri["time_sort"] = clean_fmri["SeriesTime"].fillna("00:00:00")

dedup_fmri = (
    clean_fmri.sort_values(
        by=["Subject", "Acq Date_dt", "vol_sort", "time_sort", "raw_id"]
    )
    .drop_duplicates(subset=["Subject", "Acq Date_dt"], keep="last")
    .copy()
)
print(f"Unique clean fMRI sessions (1 per Subject + Acq Date): {len(dedup_fmri)}")


Unique clean fMRI sessions (1 per Subject + Acq Date): 3250


In [10]:
# Discard I249406 and I1248413. Keep I1011356 and I424806. Makes sense when you look at the folders yourself too! nice.

In [11]:
dedup_fmri[dedup_fmri["Image Data ID"] == "I424806"]

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,ScannerModel,MagneticFieldStrength,Acceleration,NumberVolumes,SlicesPerVolume,SliceThickness,LONIStudy,Acq Date_dt,vol_sort,time_sort
1363,I424806,031_S_4005,EMCI,M,74,v31,fMRI,Extended Resting State fMRI,5/13/2014,424806,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-05-13,140.0,00:00:00


In [12]:
dedup_fmri

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,ScannerModel,MagneticFieldStrength,Acceleration,NumberVolumes,SlicesPerVolume,SliceThickness,LONIStudy,Acq Date_dt,vol_sort,time_sort
0,I238623,002_S_0295,CN,M,90,v06,fMRI,Resting State fMRI,6/02/2011,238623,...,Intera,3.0,Unaccelerated,140.0,48.0,3.31,34310.0,2011-06-02,140.0,08:14:39
1,I303069,002_S_0295,CN,M,91,v11,fMRI,Resting State fMRI,5/10/2012,303069,...,Intera,3.0,Unaccelerated,140.0,48.0,3.31,45566.0,2012-05-10,140.0,16:02:37
4,I240811,002_S_0413,CN,F,82,v06,fMRI,Resting State fMRI,6/16/2011,240811,...,Intera,3.0,Unaccelerated,140.0,48.0,3.31,34751.0,2011-06-16,140.0,12:42:52
6,I304790,002_S_0413,CN,F,82,v11,fMRI,Resting State fMRI,5/15/2012,304790,...,Intera,3.0,Unaccelerated,140.0,48.0,3.31,45738.0,2012-05-15,140.0,16:58:24
7,I371994,002_S_0413,CN,F,83,v21,fMRI,Resting State fMRI,5/10/2013,371994,...,Intera,3.0,Unaccelerated,140.0,48.0,3.31,58370.0,2013-05-10,140.0,07:37:18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3360,I1553010,941_S_7051,CN,M,64,sc,fMRI,Axial MB rsfMRI (Eyes Open),3/07/2022,1553010,...,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,193272.0,2022-03-07,976.0,11:28:50
3361,I1588336,941_S_7074,CN,M,71,sc,fMRI,Axial MB rsfMRI (Eyes Open),5/03/2022,1588336,...,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,211723.0,2022-05-03,976.0,13:13:55
3362,I1600185,941_S_7085,MCI,F,70,sc,fMRI,Axial MB rsfMRI (Eyes Open),7/01/2022,1600185,...,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,213547.0,2022-07-01,976.0,13:38:02
3363,I1591326,941_S_7087,CN,M,67,sc,fMRI,Axial MB rsfMRI (Eyes Open),6/15/2022,1591326,...,Prisma_fit,3.0,Ultrafast,976.0,64.0,2.50,212360.0,2022-06-15,976.0,15:27:27


In [13]:
# 5. Build a combined pool of verified 3D T1w scans from BOTH MRIQC_19Oct2026.csv and smri-adni-search-result.csv
#    A) Verified T1w from MRIQC (3.0T / 2.9T 3D scans)
qc_t1 = qc_df[
    (qc_df["SeriesType"] == "T1w")
    & (qc_df["ParticipantID"].isin(dedup_fmri["Subject"].unique()))
].copy()
qc_t1 = qc_t1.rename(
    columns={
        "ParticipantID": "Subject",
        "LONIImage": "T1_Raw_ID",
        "StudyDate": "T1_Date",
        "SeriesDescription": "T1_Description",
        "LONIStudy": "T1_LONIStudy",
    }
)[["Subject", "T1_Raw_ID", "T1_Date", "T1_Description", "T1_LONIStudy", "Acceleration"]]

#    B) Verified T1w from smri-adni-search-result.csv (in case any T1w wasn't in MRIQC)
valid_t1_pat = r"MPRAGE|MP-RAGE|MP RAGE|IR-SPGR|IR-FSPGR|3D T1"
junk_t1_pat = r"Field Mapping|Calibration|B1|Scout|Localizer|ME T2|GRE|Cal |ASSET|_ND|DIS2D|_MPR_Cor|_MPR_Tra|CORONAL|RFORMAT|^ax$|^cor$"
smri_valid = smri_df[
    smri_df["Description"].str.contains(valid_t1_pat, case=False, na=False)
    & ~smri_df["Description"].str.contains(junk_t1_pat, case=False, na=False)
    & smri_df["Subject ID"].isin(dedup_fmri["Subject"].unique())
].copy()
smri_valid = smri_valid.rename(
    columns={
        "Subject ID": "Subject",
        "raw_id": "T1_Raw_ID",
        "Study Date": "T1_Date",
        "Description": "T1_Description",
    }
)
smri_valid["T1_LONIStudy"] = pd.NA
smri_valid["Acceleration"] = pd.NA
smri_valid = smri_valid[
    [
        "Subject",
        "T1_Raw_ID",
        "T1_Date",
        "T1_Description",
        "T1_LONIStudy",
        "Acceleration",
    ]
]

# Combine and deduplicate T1w pool by T1_Raw_ID (preferring MRIQC row first)
t1_pool = (
    pd.concat([qc_t1, smri_valid], ignore_index=True)
    .drop_duplicates(subset=["T1_Raw_ID"], keep="first")
    .copy()
)
t1_pool["T1_Date_dt"] = pd.to_datetime(t1_pool["T1_Date"], format="mixed")

# Tie-breaker flags for T1w selection
t1_pool["is_repeat"] = t1_pool["T1_Description"].str.contains(
    r"REPEAT|rpt|#2|#3", case=False, na=False
)
t1_pool["is_cs_hs"] = t1_pool["T1_Description"].str.contains(
    r"^CS |^HS |CS_|VWIP", case=False, na=False
)
t1_pool["is_accel"] = t1_pool["Acceleration"].isin(
    ["Accelerated", "Ultrafast"]
) | t1_pool["T1_Description"].str.contains(
    r"GRAPPA|SENSE|Accelerated|Accel|acceleration", case=False, na=False
)
t1_pool["is_nd_or_reformat"] = t1_pool["T1_Description"].str.contains(
    r"_ND|DIS2D|_MPR_Cor|_MPR_Tra|CORONAL|RFORMAT", case=False, na=False
)

In [14]:
# 6. Match each deduplicated fMRI session to its single best T1w scan
matched_rows = []
for _, f_row in dedup_fmri.iterrows():
    subj = f_row["Subject"]
    f_date = f_row["Acq Date_dt"]
    f_study = f_row["LONIStudy"]

    subj_t1 = t1_pool[t1_pool["Subject"] == subj].copy()
    if subj_t1.empty:
        continue

    # Priority 0: Exact same LONIStudy session ID (0 for same study, 1 otherwise)
    subj_t1["not_same_study"] = ~(
        subj_t1["T1_LONIStudy"].notna() & (subj_t1["T1_LONIStudy"] == f_study)
    )
    subj_t1["days_apart"] = (subj_t1["T1_Date_dt"] - f_date).abs()

    best_t1 = subj_t1.sort_values(
        by=[
            "not_same_study",
            "days_apart",
            "is_nd_or_reformat",  # <-- Ensures distortion-corrected T1w always beats _ND
            "is_repeat",
            "is_cs_hs",
            "is_accel",
            "T1_Raw_ID",
        ]
    ).iloc[0]

    row_dict = f_row.drop(labels=["Acq Date_dt", "vol_sort", "time_sort"]).to_dict()
    row_dict.update(
        {
            "T1_Image_ID": f"I{best_t1['T1_Raw_ID']}",
            "T1_Raw_ID": str(best_t1["T1_Raw_ID"]),
            "T1_Study_Date": best_t1["T1_Date_dt"].strftime("%Y-%m-%d"),
            "T1_Description": best_t1["T1_Description"],
            "Same_LONIStudy": not bool(best_t1["not_same_study"]),
            "Days_Apart": int(best_t1["days_apart"].days),
        }
    )
    matched_rows.append(row_dict)

In [15]:
master_df = pd.DataFrame(matched_rows)

In [16]:
mask_1 = master_df["T1_Raw_ID"].astype(str) == "222165"
master_df[mask_1]

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,NumberVolumes,SlicesPerVolume,SliceThickness,LONIStudy,T1_Image_ID,T1_Raw_ID,T1_Study_Date,T1_Description,Same_LONIStudy,Days_Apart
741,I222174,018_S_2155,EMCI,F,69,m03,fMRI,Resting State fMRI,3/02/2011,222174,...,140.0,48.0,3.31,31797.0,I222165,222165,2011-03-02,ax,True,0


In [17]:
mask_2 = master_df["T1_Raw_ID"].astype(str) == "1527248"
master_df[mask_2]

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,NumberVolumes,SlicesPerVolume,SliceThickness,LONIStudy,T1_Image_ID,T1_Raw_ID,T1_Study_Date,T1_Description,Same_LONIStudy,Days_Apart
1468,I1527251,035_S_7030,CN,F,56,sc,fMRI,Axial rsfMRI (Eyes Open) -phase P to A,12/13/2021,1527251,...,197.0,48.0,3.4,183864.0,I1527248,1527248,2021-12-13,Sagittal 3D FLAIR phase A-P,True,0


In [18]:
# Fix Row 741 (018_S_2155 on 3/02/2011): replace 'ax' reformat (I222165) with true MPRAGE (I222168)
mask_1 = master_df["T1_Raw_ID"].astype(str) == "222165"
master_df.loc[mask_1, ["T1_Image_ID", "T1_Raw_ID", "T1_Description"]] = [
    "I222168",
    "222168",
    "MPRAGE",
]

# Fix Row 1468 (035_S_7030 on 12/13/2021): replace mislabeled FLAIR (I1527248) with true MPRAGE (I1527249)
mask_2 = master_df["T1_Raw_ID"].astype(str) == "1527248"
master_df.loc[mask_2, ["T1_Image_ID", "T1_Raw_ID", "T1_Description"]] = [
    "I1527249",
    "1527249",
    "Accelerated Sagittal MPRAGE Phase A-P",
]

master_df.to_csv(
    "../../results/dataset_creation/clean_dedup_fmri_to_t1_master.csv", index=False
)

In [19]:
# 7. Export the updated T1w Image IDs in batches of 1,500 for LONI
unique_t1_ids = sorted(master_df["T1_Raw_ID"].unique().tolist(), key=int)
print(f"Final matched fMRI-T1w pairs:    {len(master_df)}")
print(
    f"Exact same-day T1w matches:      {(master_df['Days_Apart'] == 0).sum()} / {len(master_df)} ({(master_df['Days_Apart'] == 0).mean() * 100:.2f}%)"
)
print(f"Unique T1w Image IDs to download:   {len(unique_t1_ids)}")

for i in range(0, len(unique_t1_ids), 1500):
    batch = unique_t1_ids[i : i + 1500]
    fname = f"../../results/dataset_creation/t1_image_ids_batch_{i // 1500 + 1}.txt"
    with open(fname, "w") as f:
        f.write(",".join(batch))
    print(f"Saved {len(batch):4d} IDs -> {fname}")

Final matched fMRI-T1w pairs:    3250
Exact same-day T1w matches:      3245 / 3250 (99.85%)
Unique T1w Image IDs to download:   3246
Saved 1500 IDs -> ../../results/dataset_creation/t1_image_ids_batch_1.txt
Saved 1500 IDs -> ../../results/dataset_creation/t1_image_ids_batch_2.txt
Saved  246 IDs -> ../../results/dataset_creation/t1_image_ids_batch_3.txt


In [20]:
dedup_fmri["Subject"].nunique()

1327

In [21]:
master_df

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,NumberVolumes,SlicesPerVolume,SliceThickness,LONIStudy,T1_Image_ID,T1_Raw_ID,T1_Study_Date,T1_Description,Same_LONIStudy,Days_Apart
0,I238623,002_S_0295,CN,M,90,v06,fMRI,Resting State fMRI,6/02/2011,238623,...,140.0,48.0,3.31,34310.0,I238627,238627,2011-06-02,MPRAGE,True,0
1,I303069,002_S_0295,CN,M,91,v11,fMRI,Resting State fMRI,5/10/2012,303069,...,140.0,48.0,3.31,45566.0,I303066,303066,2012-05-10,MPRAGE,True,0
2,I240811,002_S_0413,CN,F,82,v06,fMRI,Resting State fMRI,6/16/2011,240811,...,140.0,48.0,3.31,34751.0,I240812,240812,2011-06-16,MPRAGE,True,0
3,I304790,002_S_0413,CN,F,82,v11,fMRI,Resting State fMRI,5/15/2012,304790,...,140.0,48.0,3.31,45738.0,I304793,304793,2012-05-15,MPRAGE,True,0
4,I371994,002_S_0413,CN,F,83,v21,fMRI,Resting State fMRI,5/10/2013,371994,...,140.0,48.0,3.31,58370.0,I371991,371991,2013-05-10,MPRAGE,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3245,I1553010,941_S_7051,CN,M,64,sc,fMRI,Axial MB rsfMRI (Eyes Open),3/07/2022,1553010,...,976.0,64.0,2.50,193272.0,I1553005,1553005,2022-03-07,Accelerated Sagittal MPRAGE,True,0
3246,I1588336,941_S_7074,CN,M,71,sc,fMRI,Axial MB rsfMRI (Eyes Open),5/03/2022,1588336,...,976.0,64.0,2.50,211723.0,I1588331,1588331,2022-05-03,Accelerated Sagittal MPRAGE,True,0
3247,I1600185,941_S_7085,MCI,F,70,sc,fMRI,Axial MB rsfMRI (Eyes Open),7/01/2022,1600185,...,976.0,64.0,2.50,213547.0,I1600180,1600180,2022-07-01,Accelerated Sagittal MPRAGE,True,0
3248,I1591326,941_S_7087,CN,M,67,sc,fMRI,Axial MB rsfMRI (Eyes Open),6/15/2022,1591326,...,976.0,64.0,2.50,212360.0,I1591321,1591321,2022-06-15,Accelerated Sagittal MPRAGE,True,0


In [22]:
master_df["T1_Description"].value_counts()

T1_Description
Accelerated Sagittal MPRAGE                  1145
MPRAGE                                        834
Sagittal 3D Accelerated MPRAGE                318
Accelerated Sag IR-FSPGR                      281
Accelerated Sagittal IR-FSPGR                 244
Accelerated Sagittal MPRAGE (MSV22)           112
Accelerated Sagittal MPRAGE (MSV21)            89
Accelerated Sag IR-FSPGR (MSV21)               71
Accelerated Sagittal MPRAGE Phase A-P          47
Sagittal 3D Accelerated 0 angle MPRAGE         24
Accelerated Sagittal IR-FSPGR (MSV21)          20
Sag Accel IR-FSPGR                             19
Sag IR-SPGR                                    16
Sagittal 3D Accelerated MPRAGE (MSV21)         10
ORIG Accelerated Sag IR-FSPGR                   5
MPRAGE SENSE2 SENSE                             2
ORIG Accelerated_Sag_IR-FSPGR                   2
CS Sagittal MPRAGE (MSV22)                      2
MPRAGE SENSE2                                   1
Accelerated Satittal MPRAGE        